# WF2020 Return-Preserving Bull Participation Target v1

Purpose: repair the recovery-participation target semantics so the current state-objective BC label builder consumes the intended risk/recovery targets. This card does not train or promote an actor; it writes a non-leaking three-fold target contract for the next checkpoint replay.

Baselines:
- Canonical: `v19reg_v22_budget_usage_bull_guard_v1`
- Direct comparison: `wf2020_recovery_participation_target_v1` and `recovery_participation_three_fold_checkpoint_metric_replay_v1`
- Frontier handoff: `cap10_wf2020_return_preserving_target_revision_v1`

Hypothesis: mapping v2 risk/recovery states into existing consumed BC state flags (`is_downtrend_state`, `is_rebound_cluster`) will eliminate recovery-target mismatch rows before replay. The accepted tradeoff is a higher risk/recovery slice count; promotion remains blocked until official replay passes all walk-forward gates.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import math
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'src/soxl_quant').exists():
            return candidate
    raise RuntimeError(f'repo root not found from {start}')

ROOT = find_repo_root(Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from soxl_quant.policy.policy_builder import state_objective_target_position_from_row

EXPERIMENT = 'wf2020_return_preserving_bull_participation_target_v1'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / 'runtime/model_selection/card/wf2020_recovery_participation_target_v1'
SOURCE_CONTRACT_PATH = SOURCE_CARD / 'recovery_participation_three_fold_target_contract.csv'
SOURCE_SUMMARY_PATH = SOURCE_CARD / 'summary.json'
ATTRIBUTION_SUMMARY_PATH = ROOT / 'runtime/model_selection/card/wf2020_recovery_checkpoint_failure_attribution_v1/summary.json'
FRONTIER_CONTRACT_PATH = ROOT / 'runtime/model_selection/card/cap10_wf2020_return_preserving_target_revision_v1/target_revision_contract.json'
FRONTIER_SUMMARY_PATH = ROOT / 'runtime/model_selection/card/cap10_wf2020_return_preserving_target_revision_v1/summary.json'

SIDE_CARS = {
    'target_contract': CARD / 'return_preserving_bull_participation_three_fold_target_contract.csv',
    'state_projection_audit': CARD / 'state_projection_audit.csv',
    'ignored_after_projection_rows': CARD / 'ignored_after_projection_rows.csv',
    'changed_rows_preview': CARD / 'changed_rows_preview.csv',
    'selection': CARD / 'selection.json',
    'summary': CARD / 'summary.json',
    'conclusion': CARD / 'conclusion.md',
}


def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')

print('card', rel(CARD))
print('source_contract', rel(SOURCE_CONTRACT_PATH))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


card runtime/model_selection/card/wf2020_return_preserving_bull_participation_target_v1
source_contract runtime/model_selection/card/wf2020_recovery_participation_target_v1/recovery_participation_three_fold_target_contract.csv


In [2]:
source_summary = json.loads(SOURCE_SUMMARY_PATH.read_text(encoding='utf-8'))
attribution_summary = json.loads(ATTRIBUTION_SUMMARY_PATH.read_text(encoding='utf-8'))
frontier_contract = json.loads(FRONTIER_CONTRACT_PATH.read_text(encoding='utf-8'))
frontier_summary = json.loads(FRONTIER_SUMMARY_PATH.read_text(encoding='utf-8'))

source = pd.read_csv(
    SOURCE_CONTRACT_PATH,
    parse_dates=['datetime', 'train_start', 'train_end', 'eval_start', 'eval_end'],
)
required_cols = {
    'datetime', 'fold', 'teacher_position', 'risk_target_position', 'rebound_target_position',
    'recovery_participation_target_position', 'is_downtrend_state', 'is_rebound_cluster',
    'risk_control_state_v2', 'recovery_participation_state_v2', 'drawdown_recovery_conflict_state_v2',
    'risk_only_state_v2', 'recovery_only_state_v2', 'target_delta_vs_source',
}
missing = required_cols.difference(source.columns)
if missing:
    raise ValueError(f'source contract missing required columns: {sorted(missing)}')

for col in [
    'teacher_position', 'risk_target_position', 'rebound_target_position',
    'recovery_participation_target_position', 'target_delta_vs_source', 'objective_weight',
]:
    source[col] = pd.to_numeric(source[col], errors='coerce')

print('source_rows', len(source), 'folds', sorted(source['fold'].astype(str).unique()))
print('source_decision', source_summary.get('decision'))
print('attribution_decision', attribution_summary.get('decision'))
print('frontier_next', frontier_summary.get('next_mechanism') or frontier_summary.get('summary', {}).get('recommended_next_action'))


source_rows 3731 folds ['wf_2020_train', 'wf_2022_train', 'wf_2023_2024_train']
source_decision wf2020_recovery_participation_target_ready_for_checkpoint_replay_no_promotion
attribution_decision wf2020_recovery_target_semantics_mismatch_blocks_checkpoint_replay_no_promotion
frontier_next build_return_preserving_wf2020_bull_participation_target_then_rerun_three_fold_bc_checkpoint_metric_replay


In [3]:
def bool_series(frame: pd.DataFrame, column: str) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(False, index=frame.index)
    s = frame[column]
    if s.dtype == bool:
        return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float).ne(0.0)
    return s.fillna('').astype(str).str.strip().str.lower().isin({'1', 'true', 'yes', 'y'})


def consumed_target(frame: pd.DataFrame) -> pd.Series:
    return pd.Series(
        [state_objective_target_position_from_row(row) for row in frame.to_dict('records')],
        index=frame.index,
        dtype='float64',
    )


def apply_state_projection(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.sort_values(['fold', 'datetime']).reset_index(drop=True).copy()
    out['source_is_downtrend_state'] = bool_series(out, 'is_downtrend_state')
    out['source_is_rebound_cluster'] = bool_series(out, 'is_rebound_cluster')
    out['source_drawdown_control_state'] = bool_series(out, 'drawdown_control_state')
    out['source_recovery_target_consumed_position'] = consumed_target(out)

    risk_control = bool_series(out, 'risk_control_state_v2')
    recovery = bool_series(out, 'recovery_participation_state_v2')
    conflict = bool_series(out, 'drawdown_recovery_conflict_state_v2')
    risk_only = bool_series(out, 'risk_only_state_v2') | (risk_control & ~recovery)
    recovery_only = bool_series(out, 'recovery_only_state_v2') | (recovery & ~risk_control)
    projected_risk = risk_control | conflict | risk_only
    projected_recovery = recovery_only & ~projected_risk

    desired = pd.to_numeric(out['recovery_participation_target_position'], errors='coerce').astype(float).clip(0.0, 1.0)
    out['return_preserving_target_position'] = desired

    out.loc[projected_risk, 'risk_target_position'] = desired.loc[projected_risk]
    out.loc[projected_risk, 'is_downtrend_state'] = True
    out.loc[projected_risk, 'is_rebound_cluster'] = False
    if 'drawdown_control_state' in out.columns:
        out.loc[projected_risk, 'drawdown_control_state'] = True

    out.loc[projected_recovery, 'rebound_target_position'] = desired.loc[projected_recovery]
    out.loc[projected_recovery, 'is_rebound_cluster'] = True
    out.loc[projected_recovery, 'is_downtrend_state'] = False

    out['projected_risk_consumed_state'] = projected_risk.astype(bool)
    out['projected_recovery_consumed_state'] = projected_recovery.astype(bool)
    out['target_semantics_revision'] = 'wf2020_return_preserving_consumed_state_projection_v1'
    out['target_recipe_id'] = out.get('target_recipe_id', 'unknown').astype(str) + '__consumed_state_projection_v1'
    out['target_reason'] = np.select(
        [out['projected_risk_consumed_state'], out['projected_recovery_consumed_state']],
        ['projected_risk_or_conflict_to_downtrend_target', 'projected_recovery_to_rebound_target'],
        default=out.get('target_reason', pd.Series('source_contract', index=out.index)).astype(str),
    )
    out['source_contract_id'] = out.get('source_contract_id', 'unknown').astype(str) + ',wf2020_return_preserving_bull_participation_target_v1'
    out['consumed_state_objective_target_position'] = consumed_target(out)
    out['target_diff_vs_consumed'] = out['return_preserving_target_position'] - out['consumed_state_objective_target_position']
    out['target_diff_vs_consumed_abs'] = out['target_diff_vs_consumed'].abs()
    out['consumed_target_mismatch'] = out['target_diff_vs_consumed_abs'].gt(1e-9)
    out['state_flag_changed'] = (
        bool_series(out, 'is_downtrend_state').ne(out['source_is_downtrend_state'])
        | bool_series(out, 'is_rebound_cluster').ne(out['source_is_rebound_cluster'])
        | bool_series(out, 'drawdown_control_state').ne(out['source_drawdown_control_state'])
    )
    out['eval_leakage_row'] = out['datetime'].between(out['eval_start'], out['eval_end'])
    return out

contract = apply_state_projection(source)
train_mask = contract['datetime'].between(contract['train_start'], contract['train_end'])
train_contract = contract.loc[train_mask].copy()
contract.to_csv(SIDE_CARS['target_contract'], index=False)
print('target_contract', rel(SIDE_CARS['target_contract']))
print('rows', len(contract), 'train_rows', len(train_contract))


target_contract runtime/model_selection/card/wf2020_return_preserving_bull_participation_target_v1/return_preserving_bull_participation_three_fold_target_contract.csv
rows 3731 train_rows 3731


In [4]:
audit_rows = []
for fold, g in train_contract.groupby('fold', sort=True):
    source_consumed = pd.to_numeric(g['source_recovery_target_consumed_position'], errors='coerce')
    source_target = pd.to_numeric(g['recovery_participation_target_position'], errors='coerce')
    source_mismatch = (source_target - source_consumed).abs().gt(1e-9)
    row = {
        'fold': fold,
        'train_rows': int(len(g)),
        'source_mismatch_rows': int(source_mismatch.sum()),
        'projected_mismatch_rows': int(g['consumed_target_mismatch'].sum()),
        'projected_risk_rows': int(g['projected_risk_consumed_state'].sum()),
        'projected_recovery_rows': int(g['projected_recovery_consumed_state'].sum()),
        'state_flag_changed_rows': int(g['state_flag_changed'].sum()),
        'target_delta_nonzero_rows': int(pd.to_numeric(g['target_delta_vs_source'], errors='coerce').abs().gt(1e-9).sum()),
        'downtrend_rows_after': int(bool_series(g, 'is_downtrend_state').sum()),
        'rebound_rows_after': int(bool_series(g, 'is_rebound_cluster').sum()),
        'drawdown_control_rows_after': int(bool_series(g, 'drawdown_control_state').sum()),
        'mean_return_preserving_target': float(g['return_preserving_target_position'].mean()),
        'mean_consumed_target': float(g['consumed_state_objective_target_position'].mean()),
        'eval_leakage_rows': int(g['eval_leakage_row'].sum()),
    }
    audit_rows.append(row)

audit = pd.DataFrame(audit_rows)
audit.to_csv(SIDE_CARS['state_projection_audit'], index=False)
ignored = train_contract.loc[train_contract['consumed_target_mismatch']].copy()
ignored.to_csv(SIDE_CARS['ignored_after_projection_rows'], index=False)
changed_preview_cols = [
    'datetime', 'fold', 'teacher_position', 'risk_target_position', 'rebound_target_position',
    'return_preserving_target_position', 'consumed_state_objective_target_position',
    'target_delta_vs_source', 'is_downtrend_state', 'is_rebound_cluster', 'drawdown_control_state',
    'risk_control_state_v2', 'recovery_participation_state_v2', 'drawdown_recovery_conflict_state_v2',
    'projected_risk_consumed_state', 'projected_recovery_consumed_state', 'state_flag_changed', 'target_reason',
]
changed_preview = train_contract.loc[
    train_contract['state_flag_changed'] | pd.to_numeric(train_contract['target_delta_vs_source'], errors='coerce').abs().gt(1e-9),
    changed_preview_cols,
].copy()
changed_preview.to_csv(SIDE_CARS['changed_rows_preview'], index=False)

display(audit)
print('ignored_after_projection_rows', len(ignored))
print('changed_preview_rows', len(changed_preview))
assert int(audit['eval_leakage_rows'].sum()) == 0
assert int(audit['projected_mismatch_rows'].sum()) == 0


,fold,train_rows,source_mismatch_rows,projected_mismatch_rows,projected_risk_rows,projected_recovery_rows,state_flag_changed_rows,target_delta_nonzero_rows,downtrend_rows_after,rebound_rows_after,drawdown_control_rows_after,mean_return_preserving_target,mean_consumed_target,eval_leakage_rows
0,wf_2020_train,2470,226,0,839,172,300,949,897,285,897,0.304600,0.304600,0
1,wf_2022_train,505,0,0,0,0,0,0,234,66,49,0.348817,0.348817,0
2,wf_2023_2024_train,756,0,0,0,0,0,0,478,66,49,0.277530,0.277530,0


ignored_after_projection_rows 0
changed_preview_rows 1000


In [5]:
source_mismatch_total = int(audit['source_mismatch_rows'].sum())
projected_mismatch_total = int(audit['projected_mismatch_rows'].sum())
state_flag_changed_total = int(audit['state_flag_changed_rows'].sum())
projected_risk_total = int(audit['projected_risk_rows'].sum())
projected_recovery_total = int(audit['projected_recovery_rows'].sum())
target_delta_total = int(audit['target_delta_nonzero_rows'].sum())

def get_frontier_next(summary: dict) -> str | None:
    return summary.get('next_mechanism') or summary.get('summary', {}).get('recommended_next_action')

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': 'return_preserving_bull_participation_target_ready_for_official_checkpoint_replay_no_promotion',
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': [
        'wf2020_recovery_participation_target_v1',
        'recovery_participation_three_fold_checkpoint_metric_replay_v1',
        'cap10_wf2020_return_preserving_target_revision_v1',
    ],
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05',
    'closest_historical_success_case': 'cap10_three_fold_checkpoint_metric_replay_v2 passed wf_2022 and wf_2023_2024 while keeping MaxDD below 20%.',
    'closest_historical_failure_case': 'wf2020_recovery_checkpoint_failure_attribution_v1 found recovery target mismatch rows and wf_2020 return regression.',
    'why_not_duplicate': 'This card changes state semantics so existing BC label construction consumes the recovery/risk target positions; it is not another preview overlay or unconsumed target column.',
    'role': 'diagnose',
    'changed_layer': ['training_target_composition', 'model_selection', 'execution_target'],
    'target_window': 'wf_2020',
    'accepted_tradeoff': 'increase downtrend/rebound state labels only where v2 recovery/risk target already changed the target position; no eval rows are written as training labels.',
    'source_target_contract': rel(SOURCE_CONTRACT_PATH),
    'target_contract_path': rel(SIDE_CARS['target_contract']),
    'target_contract_rows': int(len(contract)),
    'source_mismatch_rows': source_mismatch_total,
    'projected_mismatch_rows': projected_mismatch_total,
    'state_flag_changed_rows': state_flag_changed_total,
    'projected_risk_rows': projected_risk_total,
    'projected_recovery_rows': projected_recovery_total,
    'target_delta_nonzero_rows': target_delta_total,
    'target_eval_leakage_rows': int(audit['eval_leakage_rows'].sum()),
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'checkpoint_replay_allowed_next': True,
    'next_mechanism': 'rerun_official_daily_three_fold_checkpoint_metric_replay_with_return_preserving_bull_participation_target',
    'sidecars': [path.name for path in SIDE_CARS.values()],
}
summary = dict(selection)
summary.update({
    'source_decision': source_summary.get('decision'),
    'attribution_decision': attribution_summary.get('decision'),
    'frontier_decision': frontier_summary.get('decision'),
    'frontier_next_mechanism': get_frontier_next(frontier_summary),
    'acceptance_criteria': frontier_contract.get('acceptance_criteria', {}),
    'fold_audit': audit.to_dict(orient='records'),
    'conclusion': [
        f'Source consumed-target mismatch rows dropped from {source_mismatch_total} to {projected_mismatch_total}.',
        'The output is a target contract only; no actor is promoted or retained.',
        'Next action is an official-daily three-fold checkpoint replay so dashboard line paths are available.',
    ],
})
write_json(SIDE_CARS['selection'], selection)
write_json(SIDE_CARS['summary'], summary)

conclusion_lines = [
    f'# {EXPERIMENT} conclusion',
    '',
    f"Decision: `{summary['decision']}`.",
    '',
    'What changed:',
    '- risk/conflict v2 rows are now projected into `is_downtrend_state=True` so BC consumes `risk_target_position`;',
    '- recovery-only v2 rows are now projected into `is_rebound_cluster=True` so BC consumes `rebound_target_position`;',
    '- non-v2 rows keep source state flags and targets.',
    '',
    'Evidence:',
    f'- Source mismatch rows: `{source_mismatch_total}`.',
    f'- Projected mismatch rows: `{projected_mismatch_total}`.',
    f'- State-flag changed rows: `{state_flag_changed_total}`.',
    f'- Target delta rows: `{target_delta_total}`.',
    f"- Eval leakage rows: `{int(audit['eval_leakage_rows'].sum())}`.",
    '',
    f"Output target contract: `{rel(SIDE_CARS['target_contract'])}`.",
    '',
    'Promotion remains blocked. Next action: official-daily three-fold checkpoint replay with this contract.',
]
SIDE_CARS['conclusion'].write_text('\n'.join(conclusion_lines).rstrip() + '\n', encoding='utf-8')
print(json.dumps(json_safe(summary), indent=2, allow_nan=False)[:5000])


{
  "experiment": "wf2020_return_preserving_bull_participation_target_v1",
  "created_at": "2026-06-14T08:01:33.642194+00:00",
  "decision": "return_preserving_bull_participation_target_ready_for_official_checkpoint_replay_no_promotion",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": [
    "wf2020_recovery_participation_target_v1",
    "recovery_participation_three_fold_checkpoint_metric_replay_v1",
    "cap10_wf2020_return_preserving_target_revision_v1"
  ],
  "reference_baseline": "soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05",
  "closest_historical_success_case": "cap10_three_fold_checkpoint_metric_replay_v2 passed wf_2022 and wf_2023_2024 while keeping MaxDD below 20%.",
  "closest_historical_failure_case": "wf2020_recovery_checkpoint_failure_attribution_v1 found recovery target mismatch rows and wf_2020 return regression.",
  "why_not_duplicate": "This card changes state semantics so existing BC label construction consumes

## Conclusion

This target card is ready only if the executed audit shows zero projected mismatch rows and zero eval leakage rows. It still does not promote or retain an actor; it only hands off a replayable target contract that the existing BC label builder can consume.